# 1. Preparing your data

`clashless` schedules presentations without double-booking anyone. Before it can help, it needs
three small structures describing your conference:

| Structure | What it says |
|---|---|
| **participants** | who's involved in each presentation |
| **sessions** | which parallel room/track each presentation belongs to |
| **unavailability** | who can't present at certain times |

This tutorial builds a tiny example of each and saves them as CSV files — the same files
[tutorial 2](02_creating_a_schedule.ipynb) uses to actually build a schedule.

## Participants and sessions

Each presentation has a unique id, a list of **participants** (unique within that presentation —
no repeats), and a **session**: a plain label identifying which parallel room/track it belongs
to. There's no separate "chair" role any more — whoever moderates is just one of the
participants, like anyone else. Two presentations sharing the same session label can never be
scheduled at the same time, since a room can't host two things at once; presentations in
*different* sessions can run in parallel freely.

Let's describe six presentations for a small conference, split across two sessions.

In [1]:
import pathlib

data_dir = pathlib.Path("data/small_conference")
data_dir.mkdir(parents=True, exist_ok=True)

participants = {
    "p1": ["Alice Kim", "Daniel Ortiz", "Grace Liu"],
    "p2": ["Ben Souza", "Grace Liu", "Henry Park", "Maria Novak"],
    "p3": ["Chloe Dubois", "Ian Ferreira", "Henry Park", "Maria Novak"],
    "p4": ["David Kaur", "Daniel Ortiz", "Ian Ferreira", "Maria Novak"],
    "p5": ["Elena Popescu", "Grace Liu", "Ian Ferreira", "Daniel Ortiz"],
    "p6": ["Farid Hossain", "Henry Park", "Grace Liu", "Daniel Ortiz"],
}
sessions = {
    "p1": "Daniel Ortiz",
    "p2": "Maria Novak",
    "p3": "Maria Novak",
    "p4": "Maria Novak",
    "p5": "Daniel Ortiz",
    "p6": "Daniel Ortiz",
}
participants

{'p1': ['Alice Kim', 'Daniel Ortiz', 'Grace Liu'],
 'p2': ['Ben Souza', 'Grace Liu', 'Henry Park', 'Maria Novak'],
 'p3': ['Chloe Dubois', 'Ian Ferreira', 'Henry Park', 'Maria Novak'],
 'p4': ['David Kaur', 'Daniel Ortiz', 'Ian Ferreira', 'Maria Novak'],
 'p5': ['Elena Popescu', 'Grace Liu', 'Ian Ferreira', 'Daniel Ortiz'],
 'p6': ['Farid Hossain', 'Henry Park', 'Grace Liu', 'Daniel Ortiz']}

Daniel Ortiz moderates `p1` (its session is named after him) while *also* being one of `p1`'s
participants — that's fine, since there's no distinguished chair role any more, just people. What
isn't allowed is a person appearing twice in the *same* presentation's participant list —
`Participants` raises a `ValueError` if you try:

In [2]:
import clashless as cl

try:
    cl.Participants({"bad": ["Alice", "Alice"]})
except ValueError as error:
    print(error)

Presentation 'bad' has a duplicate participant.


`clashless` itself only ever works with plain Python dicts, never files directly — reading a
CSV is your own job, not `clashless`'s. Since a presentation's participants are a variable-length
list, a natural file format is long: one row per (id, participant) pair. Let's save it that way
with `pandas`, then load it back and wrap it with `cl.Participants`.

In [3]:
import pandas as pd

participants_frame = (
    pd.Series(participants, name="participant").rename_axis("id").explode()
)
participants_frame.to_csv(data_dir / "participants.csv")

loaded_participants = cl.Participants(
    pd.read_csv(data_dir / "participants.csv")
    .groupby("id", sort=False)["participant"]
    .agg(list)
    .to_dict()
)
loaded_participants.data

{'p1': ['Alice Kim', 'Daniel Ortiz', 'Grace Liu'],
 'p2': ['Ben Souza', 'Grace Liu', 'Henry Park', 'Maria Novak'],
 'p3': ['Chloe Dubois', 'Ian Ferreira', 'Henry Park', 'Maria Novak'],
 'p4': ['David Kaur', 'Daniel Ortiz', 'Ian Ferreira', 'Maria Novak'],
 'p5': ['Elena Popescu', 'Grace Liu', 'Ian Ferreira', 'Daniel Ortiz'],
 'p6': ['Farid Hossain', 'Henry Park', 'Grace Liu', 'Daniel Ortiz']}

`sessions` is simpler — one label per presentation — so a plain two-column CSV round-trips it
directly via `pandas`'s index:

In [4]:
pd.Series(sessions, name="session").rename_axis("id").to_csv(data_dir / "sessions.csv")

loaded_sessions = cl.Sessions(
    pd.read_csv(data_dir / "sessions.csv", index_col="id")["session"].to_dict()
)
loaded_sessions.data

{'p1': 'Daniel Ortiz',
 'p2': 'Maria Novak',
 'p3': 'Maria Novak',
 'p4': 'Maria Novak',
 'p5': 'Daniel Ortiz',
 'p6': 'Daniel Ortiz'}

## Unavailability

This table lists exceptions: times when a specific person can't present. Each rule has a
`person`, and a `day` and `slot` that can each be left as `None` — a wildcard meaning "any value
here." That gives four kinds of rule from the same two values:

| `day` | `slot` | Meaning |
|---|---|---|
| set | `None` | unavailable **all day**, that one day |
| `None` | set | unavailable during that **slot, every day** |
| set | set | unavailable for that **one specific slot** only |
| `None` | `None` | unavailable for the **entire conference** |

`clashless` represents this as a dict keyed by person, with a list of `(day, slot)` rule tuples —
a shape close enough to write by hand for a small conference. Our example uses the first three
rule kinds:

In [5]:
unavailable_rows = [
    {"person": "Daniel Ortiz", "day": 1, "slot": None},  # all day, day 1
    {"person": "Grace Liu", "day": None, "slot": 2},  # slot 2, every day
    {"person": "Henry Park", "day": 2, "slot": 3},  # specifically day 2, slot 3
]
pd.DataFrame(unavailable_rows).to_csv(data_dir / "unavailable.csv", index=False)

loaded_frame = (
    pd.read_csv(data_dir / "unavailable.csv", dtype={"day": "Int64", "slot": "Int64"})
    .astype(object)
    .where(pd.notna, None)
)
loaded_frame["rule"] = list(zip(loaded_frame["day"], loaded_frame["slot"]))

loaded_unavailable = cl.Unavailability(
    loaded_frame.groupby("person", sort=False)["rule"].agg(list).to_dict()
)
loaded_unavailable.data

{'Daniel Ortiz': [(1, None)], 'Grace Liu': [(None, 2)], 'Henry Park': [(2, 3)]}

`Unavailability.is_unavailable(person, day, slot)` answers "is this person free at this
slot?" — it's what the solver checks internally. Here it is confirming the rules above, plus a
quick look at the fourth kind (`day` and `slot` both `None`), which we didn't add to our main
example since one person being unavailable for the *entire* conference only makes sense if they
aren't essential to any presentation.

In [6]:
def check(person, day, slot):
    """Print whether `person` is free or unavailable at (day, slot)."""
    is_unavailable = loaded_unavailable.is_unavailable(person, day, slot)
    status = "unavailable" if is_unavailable else "free"
    print(f"{person}, day {day}, slot {slot}: {status}")


check("Daniel Ortiz", 1, 1)
check("Daniel Ortiz", 2, 1)
check("Grace Liu", 3, 2)
check("Henry Park", 2, 3)
check("Henry Park", 2, 1)

everywhere_unavailable = cl.Unavailability({"Guest Speaker": [(None, None)]})
is_unavailable = everywhere_unavailable.is_unavailable("Guest Speaker", 1, 1)
print(is_unavailable, "(entire conference)")

Daniel Ortiz, day 1, slot 1: unavailable
Daniel Ortiz, day 2, slot 1: free
Grace Liu, day 3, slot 2: unavailable
Henry Park, day 2, slot 3: unavailable
Henry Park, day 2, slot 1: free
True (entire conference)


## Wrap-up

- `Participants`, `Sessions`, and `Unavailability` each wrap a plain Python `dict` — no `pandas`
  required to construct them, though it's a convenient way to round-trip a CSV (as shown above).
- A presentation's participants must be unique within that presentation — `Participants` raises a
  `ValueError` immediately if you try to repeat someone. There's no separate chair role: whoever
  moderates is just another participant, and the room/track they run is the presentation's
  `session` label instead.
- `unavailable`'s `None` day/slot combinations give you four rule types from two values.

These three files are now saved under `data/small_conference/`. Next:
[2. Creating a schedule](02_creating_a_schedule.ipynb) loads them and produces an actual
timetable.